# Inverse Modeling example (bounded mesh)

We'll show a simple example for inverse modeling using automatic differentiation on a vertex model with **bounded** (non-periodic) boundaries.

**What this notebook does:** it builds a mesh, assigns it random line tensions and equilibrates it (the *forward* problem), then uses a bilevel optimization to reshape the mesh so its aspect ratio matches a target (`cost_ratio`) -- a different, geometric-target flavour of the inverse problem from the periodic (PBC) example.

**Conventions used here** (see `docs/technical/` for the full justification):
- Mechanical/equilibrium quantities are computed in **float64** (`jax_enable_x64`).
- Unlike the periodic example, line tensions here are **not** given the additive mean-tension gauge: bounded/inverse-design runs have no analogous additive freedom to fix (tensions already enter through a `sigmoid` positivity transform, and areas are per-cell fixed targets, not inferred), so `mean_tension_gauge` is deliberately left at its default (`None`) for this driver.

First, let's check that VertAX is correctly installed by importing all we'll need for the next parts.

In [1]:
from vertax import BoundedBilevelOptimizer, BoundedMesh
# We'll see how to those important classes.

**Float64.** VertAX's mechanical equilibrium and inverse-problem machinery is validated in float64; JAX defaults to float32. Enable float64 before any mesh or array is created.

In [2]:
import jax

jax.config.update("jax_enable_x64", True)


Now, let's create our initial mesh. It will have 20 cells, in a square domain.

In [3]:
import math

# Settings
n_cells = 20
# Square domain that will contain the seeds for the mesh creation
width = math.sqrt(n_cells)
height = width

# Create a new mesh with bounded boundaries, from an initial Voronoi graph with random seeds positions.
mesh = BoundedMesh.from_random_seeds(nb_seeds=n_cells, width=width, height=height, random_key=2)



In [4]:
from vertax import plot_mesh

plot_mesh(mesh, title="Initial mesh")

print(f"Created mesh has {mesh.nb_faces} cells, {mesh.nb_edges} edges and {mesh.nb_vertices} vertices.")

Created mesh has 20 cells, 57 edges and 38 vertices.


/Users/Turlier/TURLIERLAB Dropbox/Herve Turlier/Research/projects/vertAX/vertax/src/vertax/meshes/plot.py:613: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


The mesh is associated to parameters on vertices, (half-)edges and faces (cells). The meaning of these parameters is "given" by the loss functions.

In our example, we'll only focus on the parameters of the edges, which will be the tensions at given edge ; and the parameters for faces will be their area, fixed to 0.6.

In [5]:
import jax  # VertAX uses jax and manipulates jax arrays.
import jax.numpy as jnp  # Use it like numpy

# Set mesh parameters
# 1) We don't use vertices, basic initialization.
mesh.vertices_params = jnp.asarray([0.0 for _ in range(mesh.nb_vertices)])

# 2) We give a random tension to each edge.
key = jax.random.PRNGKey(3)  # change the seed for different results
mesh.edges_params = 1 + jax.nn.sigmoid(jax.random.uniform(key, (mesh.nb_edges,))* 20 - 10)
# As we have half-edges, this part is necessary so that corresponding half-edges have the same tension.
# mesh.edges_params = jnp.repeat(he_params, 2)

# 3) We set the faces areas to 0.6.
mesh.faces_params = jnp.asarray([0.6 for _ in range(mesh.nb_faces)])

It is time to define our inner loss function. This energy function penalizes cell areas different than 0.6 and the length of edges depending on the tension on them.

We use the library's `make_energy_line_tensions_bounded` factory (rather than hand-writing this energy) so `K_areas` is explicit and the convention matches every bounded driver/test in the repository.

In [6]:
from vertax.energy import make_energy_line_tensions_bounded

# K_areas sets the relative stiffness of the area term against the line-tension term. It is NOT
# hardcoded inside vertax.energy -- it must be passed explicitly and asserted at every call site
# (a past hardcoded value was a reproducibility hazard). The bounded convention differs from the
# periodic one: every bounded driver/test in this repository (see tests/test_*_bounded.py) uses
# K_areas = 40 (an older hardcoded bounded energy used an equivalent effective K = 40 but wrote it
# as a prefactor of 20 with no explicit 0.5 -- same physics, different-looking constant).
K_areas = 40.0
assert K_areas == 40.0  # explicit per the project's standing numerical rule

# `sigmoid_tensions=True` reproduces this notebook's convention of drawing `edges_params` already
# passed through a sigmoid (see the previous cell): the energy applies the same transform internally
# so `he_params` is consistently "pre-sigmoid" wherever it is read. It is opt-in (default False)
# because it changes what `he_params` physically means; report it explicitly whenever used.
energy = make_energy_line_tensions_bounded(K_areas, sigmoid_tensions=True)


Now we are ready to perform the inner optimization ! This part is done by a BoundedBilevelOptimizer, that stores the hyper-parameters for the optimizations.

Here are some parameters that are taken into account with the inner optimization :

In [7]:
import optax  # Defines solvers for the optimization problems

bilevel_optimizer = BoundedBilevelOptimizer()
bilevel_optimizer.loss_function_inner = energy  # Don't forget to set the loss function !
# Note: those are base values so the following can be omitted
bilevel_optimizer.inner_solver = optax.sgd(learning_rate=0.01)  # inner solver
bilevel_optimizer.update_T1 = True # Perform T1 transitions if necessary.
bilevel_optimizer.min_dist_T1 = 0.005  # Threshold to perform the T1 transitions.
bilevel_optimizer.max_nb_iterations = 1000  # Upper bound for the number of minimization operations
bilevel_optimizer.tolerance = 1e-4  # Below this level, we consider that the loss is stagnating
bilevel_optimizer.patience = 5  # Maximum number of consecutive stagnating loss before we stop.

Let's perform the inner optimization. It's as simple as that:

In [8]:
# Another parameter is beta (for EP).
loss_history = bilevel_optimizer.inner_optimization(mesh)
# If you want to select only a subset of vertices, edges, and faces, it's possible:
# bilevel_optimizer.inner_optimization(mesh,
#     only_on_vertices=[list_vertex_ids],
#     only_on_edges=[list_edges_id],
#     only_on_faces=[list_faces_id],
# )

In [9]:
plot_mesh(mesh, title="The mesh after the inner optimization.")

/Users/Turlier/TURLIERLAB Dropbox/Herve Turlier/Research/projects/vertAX/vertax/src/vertax/meshes/plot.py:613: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


The inner optimization nudged the mesh to minimize the length of edges with high tension while keeping the area of each face close to 0.6.

We'll use a bilevel optimization method now.

In addition to the previous parameters for the inner optimization, the following parameters are used during the bilevel optimization process:

In [10]:
from vertax.cost import cost_ratio  # We use a loss function defined by vertax, but you can use yours.

# This cost defines an approximate ratio between the long and short axes of our cellular cluster ;
# and compares the current ratio against a fixed target of 2.0,
# which corresponds to the goal to be achieved in this particular experiment.
bilevel_optimizer.loss_function_outer = cost_ratio

from vertax import BilevelOptimizationMethod  # An enumeration of the methods you can use

# Once again this part can be omitted because those are the base value for these parameters
bilevel_optimizer.outer_solver = optax.adam(learning_rate=0.0001, nesterov=True)  # outer solver
bilevel_optimizer.bilevel_optimization_method = BilevelOptimizationMethod.EQUILIBRIUM_PROPAGATION

Let's run our bilevel optimization process. The first epoch is always a bit long but the next ones will be quick.

In a real simulation, a high number of epochs is necessary for a good result. Here, for this basic tutorial, we'll only use a few epochs.

(Also note that the performances are always worse in a notebook !)

In [11]:
from time import perf_counter

nb_epochs = 10

cost_history = []
for epoch in range(nb_epochs + 1):
    t_start = perf_counter()
    # This cost defines an approximate ratio between the long and short axes of our cellular cluster ;
    # and compares the current ratio against a fixed target of 2.0,
    # which corresponds to the goal to be achieved in this particular experiment.
    cost = cost_ratio(mesh.vertices)
    cost_history.append(cost)
    print(f"epoch: {epoch}/{nb_epochs}, actual cost is {cost}")

    loss_history = bilevel_optimizer.bilevel_optimization(mesh)
    print(f"epoch {epoch} took {perf_counter() - t_start} seconds.")

[vertax] WARNING: beta was never set; EQUILIBRIUM_PROPAGATION is using the default beta=0.01. Set it explicitly and record it with your results -- note the manuscript convention is 2x the code value.


epoch: 0/10, actual cost is 0.7543922662734985


epoch 0 took 7.413944665808231 seconds.
epoch: 1/10, actual cost is 0.7538479566574097


epoch 1 took 0.9717224170453846 seconds.
epoch: 2/10, actual cost is 0.7532450556755066


epoch 2 took 0.9599983752705157 seconds.
epoch: 3/10, actual cost is 0.752680242061615


epoch 3 took 0.9549499996937811 seconds.
epoch: 4/10, actual cost is 0.752166748046875


... (3 outputs left out)

epoch 7 took 0.954120249953121 seconds.
epoch: 8/10, actual cost is 0.7505723237991333


epoch 8 took 0.9676925828680396 seconds.
epoch: 9/10, actual cost is 0.7502637505531311


epoch 9 took 0.9608636251650751 seconds.
epoch: 10/10, actual cost is 0.7499808669090271


epoch 10 took 0.9443618752993643 seconds.


Just to check that the process is working, let's do a basic plot of the cost in function of the epoch.

In [12]:
import matplotlib.pyplot as plt

plt.plot(range(nb_epochs+1), cost_history)

Of course, a higher number of epochs is necessary for the cost function to stabilize. If we plot the mesh after a sufficient number of epochs, we'll see that the vertices moved to match the tensions and area constraints.

In [13]:
plot_mesh(mesh, title=f"The mesh after {nb_epochs} epochs")

/Users/Turlier/TURLIERLAB Dropbox/Herve Turlier/Research/projects/vertAX/vertax/src/vertax/meshes/plot.py:613: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


We can also plot the tensions and areas:

In [14]:
from vertax import EdgePlot, FacePlot, VertexPlot  # Enumerations to decide what to show on the plot.

plot_mesh(mesh, edge_plot=EdgePlot.EDGE_PARAMETER, edge_parameters_name="tension",
          face_plot=FacePlot.FACE_PARAMETER, face_parameters_name="area",
          vertex_plot=VertexPlot.INVISIBLE,
          title="Area of cells and tensions on edges.")